In [ ]:
%cd /content

!pip install unsloth trl transformers accelerate peft bitsandbytes wandb matplotlib datasets loguru fastapi uvicorn httpx -q
!pip install git+https://github.com/meta-pytorch/OpenEnv.git -q

!rm -rf /content/ProcureRL
!git clone https://github.com/ShivenduShivu/ProcureRL.git /content/ProcureRL
%cd /content/ProcureRL
!pip install -e . --no-deps --no-build-isolation -q


In [ ]:
%cd /content/ProcureRL

import sys
if "/content/ProcureRL" not in sys.path:
    sys.path.insert(0, "/content/ProcureRL")

import json
import os
import matplotlib.pyplot as plt
import torch
from IPython.display import clear_output
from datasets import Dataset
from transformers import TrainerCallback
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

from agenticpay.openenv_adapter.procure_env_extended import ProcureEnvExtended
from training.prompt_builder import format_for_trl
from training.evaluate import evaluate_model, print_evaluation_report
from training.plot_results import save_before_after_plot, save_training_plots


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)


In [ ]:
print('Running BASELINE evaluation...')
baseline_metrics = evaluate_model(model, tokenizer, n_episodes=30, difficulty='easy')
print_evaluation_report(baseline_metrics, label='BASELINE (Before Training)')
os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)


In [ ]:
import hashlib
from agenticpay.openenv_adapter.action import BuyerAction

env_for_reward = ProcureEnvExtended(difficulty='easy')
reward_component_log = []

def compute_reward_for_grpo(prompts, completions, **kwargs):
    rewards = []
    batch_components = {
        "reward/total": [], "reward/savings": [], 
        "reward/efficiency": [], "reward/compliance": [],
        "reward/deal_quality": [], "reward/penalty_constraint": [],
        "reward/penalty_timeout": [], "metric/deal_reached": [],
        "metric/constraint_violated": [],
    }
    for prompt, completion in zip(prompts, completions):
        try:
            stable_seed = int(
                hashlib.md5(prompt.encode()).hexdigest()[:8], 16
            ) % 10000
            obs, _ = env_for_reward.reset(seed=stable_seed)
            
            # Parse the offered price before stepping
            buyer_action = BuyerAction.from_text(completion)
            
            # Step with completion
            obs, _, terminated, truncated, _ = env_for_reward.step(completion)
            
            # Use shaped reward instead of flat step reward
            shaped_reward = env_for_reward.get_shaped_reward(
                buyer_action.offered_price,
                action_text=completion
            )
            rewards.append(float(shaped_reward))
            
            breakdown = getattr(env_for_reward, '_last_reward_breakdown', None)
            breakdown_dict = breakdown.to_dict() if breakdown else {}
            for key in batch_components:
                batch_components[key].append(
                    float(breakdown_dict.get(key, 0.0))
                )
        except Exception:
            rewards.append(-0.3)
            for key in batch_components:
                batch_components[key].append(0.0)
    
    reward_component_log.append({
        key: sum(v)/len(v) if v else 0.0
        for key, v in batch_components.items()
    })
    return rewards



In [ ]:
env_for_data = ProcureEnvExtended(difficulty="easy")
training_prompts = []
for i in range(500):
    obs, _ = env_for_data.reset(seed=i)
    messages = format_for_trl(obs)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    training_prompts.append({"prompt": prompt})
train_dataset = Dataset.from_list(training_prompts)
print("dataset size:", len(train_dataset))


In [ ]:
training_log = []

class LiveRewardCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step == 0:
            return control

        latest = reward_component_log[-1] if reward_component_log else {}
        entry = {
            "step": int(state.global_step),
            "reward/total": float(latest.get("reward/total", logs.get("reward", 0.0))),
            "reward/savings": float(latest.get("reward/savings", 0.0)),
            "reward/efficiency": float(latest.get("reward/efficiency", 0.0)),
            "reward/compliance": float(latest.get("reward/compliance", 0.0)),
            "reward/deal_quality": float(latest.get("reward/deal_quality", 0.0)),
            "trainer/reward": float(logs.get("reward", latest.get("reward/total", 0.0))),
            "trainer/reward_std": float(logs.get("reward_std", 0.0)),
            "trainer/kl": float(logs.get("kl", 0.0)),
        }
        training_log.append(entry)

        clear_output(wait=True)
        print(f"Training step {state.global_step}/{int(args.max_steps)}")
        print(f"Latest shaped reward mean: {entry["reward/total"]:.4f}")
        print(f"Trainer reward std: {entry["trainer/reward_std"]:.4f}")
        print(f"KL: {entry["trainer/kl"]:.4f}")

        steps = [item["step"] for item in training_log]
        rewards = [item["reward/total"] for item in training_log]
        reward_std = [item["trainer/reward_std"] for item in training_log]
        kl_values = [item["trainer/kl"] for item in training_log]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(steps, rewards, color="#1d4ed8", linewidth=2)
        axes[0].axhline(0.0, color="#94a3b8", linestyle="--", linewidth=1)
        axes[0].set_title("Mean Episode Reward")
        axes[0].set_xlabel("Training Step")
        axes[0].set_ylabel("Mean Episode Reward")
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(steps, reward_std, color="#d97706", linewidth=2)
        axes[1].set_title("Reward Std")
        axes[1].set_xlabel("Training Step")
        axes[1].set_ylabel("Reward Std")
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(steps, kl_values, color="#059669", linewidth=2)
        axes[2].set_title("KL Divergence")
        axes[2].set_xlabel("Training Step")
        axes[2].set_ylabel("KL")
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        return control

grpo_config = GRPOConfig(
    output_dir="./procurerl_checkpoints",
    max_steps=150,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=8,
    max_completion_length=80,
    temperature=0.8,
    logging_steps=5,
    save_steps=100,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[compute_reward_for_grpo],
    args=grpo_config,
    train_dataset=train_dataset,
)
trainer.add_callback(LiveRewardCallback())

trainer.train()

print("logged training steps:", len(training_log))


In [ ]:
print('Running POST-TRAINING evaluation...')
trained_metrics = evaluate_model(model, tokenizer, n_episodes=30, difficulty='easy')
print_evaluation_report(trained_metrics, label='TRAINED (After GRPO)')
with open('results/trained_metrics.json', 'w') as f:
    json.dump(trained_metrics, f, indent=2)


In [ ]:
os.makedirs("results/plots", exist_ok=True)
save_training_plots(training_log, output_dir="results/plots")
save_before_after_plot(baseline_metrics, trained_metrics, output_dir="results/plots")
print("Improvement in deal rate:", trained_metrics["deal_rate"] - baseline_metrics["deal_rate"])
print("Improvement in mean reward:", trained_metrics["mean_episode_reward"] - baseline_metrics["mean_episode_reward"])
print("Improvement in mean savings:", trained_metrics["mean_savings"] - baseline_metrics["mean_savings"])
print("Improvement in constraint violations:", trained_metrics["constraint_violation_rate"] - baseline_metrics["constraint_violation_rate"])


In [ ]:
model.save_pretrained_merged(
    'procurerl_model_merged',
    tokenizer,
    save_method='merged_16bit',
)
print('Model saved.')
